# Module 3: Preprocessing & Dataset Splits

## 3.1 · Preprocessing configuration

In [ ]:
PROJECT_DIR = "CounterShield"
DATASETS = ["bot_iot", "cic_ids2017", "unsw_nb15"]
SEED = 42

SPLIT = {
    "bot_iot":     {"use_official_test": False, "test_frac": 0.15, "val_frac": 0.15, "seed": SEED},
    "cic_ids2017": {"use_official_test": False, "test_frac": 0.15, "val_frac": 0.15, "seed": SEED},
    "unsw_nb15":   {"use_official_test": True,  "test_frac": None, "val_frac": 0.15, "seed": SEED},
}
PREP = {
    "numeric_scaler": "quantile",
    "quantile_subsample": 200_000,
    "max_categories": 20,
    "seed": SEED,
}

### Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import sys, datetime as dt
try:
    from IPython.display import display
except ImportError:
    display = print

import os, re, gc, json, time, hashlib, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold

TARGET_COLS = ["y_binary", "y_class", "y_family", "y_subclass"]

def sha256_file(path, block=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

def human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024: return f"{n:,.1f} {u}"
        n /= 1024
    return f"{n:,.1f} PB"

def _strat_labels(y):
    y = pd.Series(y).astype(str)
    vc = y.value_counts()
    small = vc.index[vc < 4]
    return y.where(~y.isin(small), "__rare__").to_numpy()

def make_split_manifest(df, key, cfg):
    idx = np.arange(len(df))
    y = df["y_class"].astype(str).to_numpy()
    if cfg["use_official_test"]:
        is_test = (df["split"].astype(str) == "test").to_numpy()
        te, pool = idx[is_test], idx[~is_test]
        tr, va = train_test_split(pool, test_size=cfg["val_frac"], stratify=_strat_labels(y[pool]), random_state=cfg["seed"])
    else:
        trva, te = train_test_split(idx, test_size=cfg["test_frac"], stratify=_strat_labels(y), random_state=cfg["seed"])
        tr, va = train_test_split(trva, test_size=cfg["val_frac"] / (1 - cfg["test_frac"]),
                                  stratify=_strat_labels(y[trva]), random_state=cfg["seed"])
    split = np.empty(len(df), dtype=object)
    split[tr], split[va], split[te] = "train", "val", "test"
    man = pd.DataFrame({"row": idx, "split": split, "y_binary": df["y_binary"].to_numpy(), "y_class": y})
    return man

def split_summary(man, key):
    t = man.groupby(["split", "y_class"]).size().unstack(0, fill_value=0)
    t = t.reindex(columns=[c for c in ["train", "val", "test"] if c in t.columns])
    prop = t / t.sum()
    out = t.copy()
    for s in t.columns:
        out[f"{s}_%"] = (100 * prop[s]).round(3)
    out.insert(0, "dataset", key)
    return out.reset_index()

def build_preprocessor(num, cat, n_train, cfg):
    if cfg["numeric_scaler"] == "quantile":
        scaler = QuantileTransformer(output_distribution="normal", n_quantiles=int(min(1000, n_train)),
                                     subsample=cfg["quantile_subsample"], random_state=cfg["seed"])
    elif cfg["numeric_scaler"] == "robust":
        scaler = RobustScaler()
    else:
        scaler = StandardScaler()
    parts = [("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", scaler)]), num)]
    if cat:
        parts.append(("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=cfg["max_categories"],
                                     sparse_output=False, dtype=np.float32))]), cat))
    ct = ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=True)
    return Pipeline([("columns", ct), ("drop_constant", VarianceThreshold(0.0))])

def _prep_X(df, feats, cat):
    X = df[feats].copy()
    for c in cat:
        X[c] = X[c].astype(str)
    return X

def fit_preprocessor(clean, man, schema, cfg):
    feats, cat = schema["features"], schema["categorical_features"]
    num = [c for c in feats if c not in cat]
    tr_rows = man.loc[man.split == "train", "row"].to_numpy()
    Xtr = _prep_X(clean.iloc[tr_rows], feats, cat)
    pre = build_preprocessor(num, cat, len(Xtr), cfg)
    t0 = time.time()
    pre.fit(Xtr)
    info = {"fit_rows": int(len(Xtr)), "fit_seconds": round(time.time() - t0, 1),
            "train_rows_sha256": hashlib.sha256(np.ascontiguousarray(tr_rows).tobytes()).hexdigest(),
            "n_input_features": len(feats), "n_numeric": len(num), "n_categorical": len(cat),
            "n_output_features": int(len(pre.get_feature_names_out())),
            "dropped_constant_in_train": [str(f) for f, keep in zip(pre["columns"].get_feature_names_out(), pre["drop_constant"].get_support()) if not keep]}
    return pre, info

def validate_preprocessor(pre, info, clean, man, schema, cfg, sample=200_000):
    feats, cat = schema["features"], schema["categorical_features"]
    rng = np.random.default_rng(cfg["seed"])
    res = []
    def add(name, ok, detail=""):
        res.append({"check": name, "passed": bool(ok), "detail": str(detail)})
    tr = man.loc[man.split == "train", "row"].to_numpy()
    va = man.loc[man.split == "val", "row"].to_numpy()
    add("fitted on the training split only", info["fit_rows"] == len(tr), f"{info['fit_rows']:,} rows fitted, train has {len(tr):,}")
    add("input columns match schema", list(pre["columns"].feature_names_in_) == feats)
    s_tr = rng.choice(tr, min(sample, len(tr)), replace=False)
    s_va = rng.choice(va, min(sample, len(va)), replace=False)
    Ztr = pre.transform(_prep_X(clean.iloc[s_tr], feats, cat))
    Zva = pre.transform(_prep_X(clean.iloc[s_va], feats, cat))
    names = pre.get_feature_names_out()
    add("output width = feature names", Ztr.shape[1] == len(names) == Zva.shape[1], f"{len(names)} features")
    add("no NaN/inf (train sample)", np.isfinite(Ztr).all())
    add("no NaN/inf (validation sample)", np.isfinite(Zva).all())
    numcols = [i for i, n in enumerate(names) if str(n).startswith("num__")]
    if numcols:
        A, B = Ztr[:, numcols], Zva[:, numcols]
        if cfg["numeric_scaler"] == "quantile":
            lim = float(np.abs(A).max())
            add("numeric outputs bounded (quantile-normal)", lim <= 5.3, f"max |z| {lim:.2f}")
        else:
            m = np.abs(A.mean(axis=0)); sd = A.std(axis=0)
            add("numeric features centred and scaled (train)", np.median(m) < 0.25 and 0.3 < np.median(sd) < 3,
                f"median |mean| {np.median(m):.3f}, median std {np.median(sd):.3f}")
        dm = np.abs(A.mean(axis=0) - B.mean(axis=0)); ds = np.abs(A.std(axis=0) - B.std(axis=0))
        add("validation distribution matches train", np.median(dm) < 0.1 and np.median(ds) < 0.1,
            f"median |mean diff| {np.median(dm):.3f}, median |std diff| {np.median(ds):.3f}")
    if cat:
        probe = _prep_X(clean.iloc[s_va[:50]], feats, cat)
        for c in cat: probe[c] = "__never_seen__"
        try:
            z = pre.transform(probe); add("unseen categories handled", np.isfinite(z).all())
        except Exception as e:
            add("unseen categories handled", False, repr(e)[:150])
    add("deterministic transform", np.array_equal(pre.transform(_prep_X(clean.iloc[s_va[:1000]], feats, cat)), Zva[:min(1000, len(Zva))]))
    with tempfile.TemporaryDirectory() as d:
        p = Path(d) / "pre.joblib"; joblib.dump(pre, p); pre2 = joblib.load(p)
        add("save/load round trip identical", np.array_equal(pre2.transform(_prep_X(clean.iloc[s_va[:1000]], feats, cat)), Zva[:min(1000, len(Zva))]))
    return pd.DataFrame(res)

def transform_splits(pre, clean, man, schema, out_dir, chunk=500_000):
    feats, cat = schema["features"], schema["categorical_features"]
    names = [str(n) for n in pre.get_feature_names_out()]
    labels = [c for c in TARGET_COLS if c in clean.columns]
    out_dir.mkdir(parents=True, exist_ok=True)
    info = {}
    for s in ["train", "val", "test"]:
        rows = man.loc[man.split == s, "row"].to_numpy()
        parts = []
        for i in range(0, len(rows), chunk):
            r = rows[i:i + chunk]
            Z = pre.transform(_prep_X(clean.iloc[r], feats, cat)).astype(np.float32)
            part = pd.DataFrame(Z, columns=names)
            for c in labels:
                part[c] = clean[c].iloc[r].to_numpy()
            part["row"] = r
            parts.append(part)
        d = pd.concat(parts, ignore_index=True)
        for c in ("y_class", "y_family", "y_subclass"):
            if c in d.columns: d[c] = d[c].astype(str)
        f = out_dir / f"{s}.parquet"
        d.to_parquet(f, index=False)
        info[s] = {"path": f, "rows": int(len(d)), "cols": int(d.shape[1])}
        print(f"  {s:<5} {len(d):>10,} rows x {len(names)} features -> {f.name} ({human(f.stat().st_size)})")
        del parts, d; gc.collect()
    return names, labels, info

def final_validation(key, man, names, labels, info, schema, split_cfg, tol=0.01):
    res = []
    def add(name, ok, detail=""):
        res.append({"dataset": key, "check": name, "passed": bool(ok), "detail": str(detail)})
    data = {s: pd.read_parquet(info[s]["path"]) for s in ["train", "val", "test"]}
    for s, d in data.items():
        add(f"{s}: row count matches manifest", len(d) == int((man.split == s).sum()), f"{len(d):,}")
        X = d[names].to_numpy()
        add(f"{s}: no NaN/inf", np.isfinite(X).all())
        add(f"{s}: features stored as float32", all(d[n].dtype == np.float32 for n in names))
    add("same columns in every split", len({tuple(d.columns) for d in data.values()}) == 1)
    forbidden = set(TARGET_COLS) | {"split", "source_dataset", "source_file"} | set(schema.get("meta_columns", []))
    leaked = [n for n in names if any(n.split("__", 1)[-1] == f or n.split("__", 1)[-1].startswith(f + "_") for f in forbidden)]
    add("no label/meta columns among features", not leaked, ", ".join(leaked))
    rows = {s: set(d["row"]) for s, d in data.items()}
    add("splits are disjoint", not (rows["train"] & rows["val"] or rows["train"] & rows["test"] or rows["val"] & rows["test"]))
    add("splits cover every cleaned row", sum(map(len, rows.values())) == len(man))
    tr_classes = set(data["train"]["y_class"])
    missing = (set(data["val"]["y_class"]) | set(data["test"]["y_class"])) - tr_classes
    add("every val/test class appears in train", not missing, ", ".join(sorted(missing)))
    ptr = data["train"]["y_class"].value_counts(normalize=True)
    for s in (["val"] if split_cfg["use_official_test"] else ["val", "test"]):
        p = data[s]["y_class"].value_counts(normalize=True).reindex(ptr.index, fill_value=0)
        dmax = float((p - ptr).abs().max())
        add(f"{s}: class proportions match train (stratified)", dmax < tol, f"max diff {dmax:.4f}")
    htr = set(pd.util.hash_pandas_object(data["train"][names], index=False))
    hte = pd.util.hash_pandas_object(data["test"][names], index=False)
    ov = int(hte.isin(htr).sum())
    note = {"train_test_identical_feature_rows": ov, "pct_of_test": round(100 * ov / len(hte), 2)}
    return pd.DataFrame(res), note

BASE      = Path("/content/drive/MyDrive") / PROJECT_DIR
CLEAN_DIR = BASE / "processed" / "clean"
SPLIT_DIR = BASE / "processed" / "splits"
READY_DIR = BASE / "processed" / "model_ready"
PRE_DIR   = BASE / "models" / "preprocessors"
RES_DIR   = BASE / "results" / "module3"
CONF_DIR  = BASE / "configs"
for p in (SPLIT_DIR, READY_DIR, PRE_DIR, RES_DIR, CONF_DIR): p.mkdir(parents=True, exist_ok=True)

SCHEMA = json.loads((CONF_DIR / "feature_schema.json").read_text())
(CONF_DIR / "preprocessing_config.json").write_text(json.dumps(
    {"split": SPLIT, "preprocessing": PREP, "sklearn": sklearn.__version__, "pandas": pd.__version__,
     "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}, indent=2))
CLEAN = {}
for k in DATASETS:
    CLEAN[k] = pd.read_parquet(CLEAN_DIR / f"{k}_clean.parquet")
    s = SCHEMA[k]
    print(f"{k:<12} {len(CLEAN[k]):>10,} rows | {len(s['features'])} features ({len(s['categorical_features'])} categorical: {s['categorical_features']})")
PRE, PRE_INFO, MAN, NAMES, LABELS, OUT, FINAL, OVERLAP = {}, {}, {}, {}, {}, {}, [], {}
print("sklearn", sklearn.__version__, "| config ->", (CONF_DIR / "preprocessing_config.json").relative_to(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bot_iot         903,922 rows | 20 features (3 categorical: ['flgs', 'proto', 'state'])
cic_ids2017   2,198,595 rows | 69 features (0 categorical: [])
unsw_nb15       181,600 rows | 42 features (3 categorical: ['proto', 'service', 'state'])
sklearn 1.6.1 | config -> configs/preprocessing_config.json


## 3.2 · Train / validation / test split manifests

In [ ]:
summaries = []
for k in DATASETS:
    MAN[k] = make_split_manifest(CLEAN[k], k, SPLIT[k])
    MAN[k].to_parquet(SPLIT_DIR / f"{k}_split_manifest.parquet", index=False)
    summaries.append(split_summary(MAN[k], k))
    c = MAN[k].split.value_counts()
    print(f"{k:<12} train {c['train']:>9,} | val {c['val']:>8,} | test {c['test']:>8,}   "
          f"attack rate train/val/test: " + " / ".join(f"{MAN[k].loc[MAN[k].split==s,'y_binary'].mean():.3f}" for s in ['train','val','test']))
split_tab = pd.concat(summaries, ignore_index=True)
split_tab.to_csv(RES_DIR / "split_summary.csv", index=False)
display(split_tab)

bot_iot      train   632,744 | val  135,589 | test  135,589   attack rate train/val/test: 0.990 / 0.990 / 0.990
cic_ids2017  train 1,539,015 | val  329,790 | test  329,790   attack rate train/val/test: 0.153 / 0.153 / 0.153
unsw_nb15    train    84,377 | val   14,891 | test   82,332   attack rate train/val/test: 0.480 / 0.480 / 0.551


split,y_class,dataset,train,val,test,train_%,val_%,test_%
0,benign,bot_iot,6401,1371,1371,1.012,1.011,1.011
1,ddos,bot_iot,209260,44842,44842,33.072,33.072,33.072
2,dos,bot_iot,208798,44743,44743,32.999,32.999,32.999
3,reconnaissance,bot_iot,207174,44395,44395,32.742,32.742,32.742
4,theft,bot_iot,1111,238,238,0.176,0.176,0.176
5,benign,cic_ids2017,1303688,279362,279362,84.709,84.709,84.709
6,bot,cic_ids2017,978,209,210,0.064,0.063,0.064
7,ddos,cic_ids2017,89607,19202,19202,5.822,5.822,5.822
8,dos_goldeneye,cic_ids2017,7200,1543,1543,0.468,0.468,0.468
9,dos_hulk,cic_ids2017,120882,25903,25903,7.855,7.854,7.854


## 3.3 · BoT-IoT preprocessor fitting

In [ ]:
PRE["bot_iot"], PRE_INFO["bot_iot"] = fit_preprocessor(CLEAN["bot_iot"], MAN["bot_iot"], SCHEMA["bot_iot"], PREP)
joblib.dump(PRE["bot_iot"], PRE_DIR / "bot_iot_preprocessor.joblib")
display(pd.Series({k_: v for k_, v in PRE_INFO["bot_iot"].items() if k_ != "train_rows_sha256"}, dtype=object).to_frame("bot_iot"))

,bot_iot
fit_rows,632744
fit_seconds,5.7
n_input_features,20
n_numeric,17
n_categorical,3
n_output_features,46
dropped_constant_in_train,[]


## 3.4 · BoT-IoT preprocessor validation

In [ ]:
v = validate_preprocessor(PRE["bot_iot"], PRE_INFO["bot_iot"], CLEAN["bot_iot"], MAN["bot_iot"], SCHEMA["bot_iot"], PREP)
v.insert(0, "dataset", "bot_iot"); v.to_csv(RES_DIR / "bot_iot_preprocessor_validation.csv", index=False)
display(v)
assert v.passed.all(), "preprocessor validation failed - see table"

,dataset,check,passed,detail
0,bot_iot,fitted on the training split only,True,"632,744 rows fitted, train has 632,744"
1,bot_iot,input columns match schema,True,
2,bot_iot,output width = feature names,True,46 features
3,bot_iot,no NaN/inf (train sample),True,
4,bot_iot,no NaN/inf (validation sample),True,
5,bot_iot,numeric outputs bounded (quantile-normal),True,max |z| 5.20
6,bot_iot,validation distribution matches train,True,"median |mean diff| 0.003, median |std diff| 0.001"
7,bot_iot,unseen categories handled,True,
8,bot_iot,deterministic transform,True,
9,bot_iot,save/load round trip identical,True,


## 3.5 · BoT-IoT transformation

In [ ]:
NAMES["bot_iot"], LABELS["bot_iot"], OUT["bot_iot"] = transform_splits(PRE["bot_iot"], CLEAN["bot_iot"], MAN["bot_iot"], SCHEMA["bot_iot"], READY_DIR / "bot_iot")

fv, OVERLAP["bot_iot"] = final_validation("bot_iot", MAN["bot_iot"], NAMES["bot_iot"], LABELS["bot_iot"], OUT["bot_iot"], SCHEMA["bot_iot"], SPLIT["bot_iot"])
FINAL.append(fv); fv.to_csv(RES_DIR / "bot_iot_final_validation.csv", index=False)
display(fv); print("train/test identical feature rows:", OVERLAP["bot_iot"])
assert fv.passed.all(), "final validation failed - see table"
del CLEAN["bot_iot"]; gc.collect()

  train    632,744 rows x 46 features -> train.parquet (23.1 MB)
  val      135,589 rows x 46 features -> val.parquet (6.0 MB)
  test     135,589 rows x 46 features -> test.parquet (6.1 MB)


,dataset,check,passed,detail
0,bot_iot,train: row count matches manifest,True,"632,744"
1,bot_iot,train: no NaN/inf,True,
2,bot_iot,train: features stored as float32,True,
3,bot_iot,val: row count matches manifest,True,"135,589"
4,bot_iot,val: no NaN/inf,True,
5,bot_iot,val: features stored as float32,True,
6,bot_iot,test: row count matches manifest,True,"135,589"
7,bot_iot,test: no NaN/inf,True,
8,bot_iot,test: features stored as float32,True,
9,bot_iot,same columns in every split,True,


train/test identical feature rows: {'train_test_identical_feature_rows': 3, 'pct_of_test': 0.0}


0

## 3.6 · CICIDS-2017 preprocessor fitting

In [ ]:
PRE["cic_ids2017"], PRE_INFO["cic_ids2017"] = fit_preprocessor(CLEAN["cic_ids2017"], MAN["cic_ids2017"], SCHEMA["cic_ids2017"], PREP)
joblib.dump(PRE["cic_ids2017"], PRE_DIR / "cic_ids2017_preprocessor.joblib")
display(pd.Series({k_: v for k_, v in PRE_INFO["cic_ids2017"].items() if k_ != "train_rows_sha256"}, dtype=object).to_frame("cic_ids2017"))

,cic_ids2017
fit_rows,1539015
fit_seconds,45.4
n_input_features,69
n_numeric,69
n_categorical,0
n_output_features,69
dropped_constant_in_train,[]


## 3.7 · CICIDS-2017 preprocessor validation

In [ ]:
v = validate_preprocessor(PRE["cic_ids2017"], PRE_INFO["cic_ids2017"], CLEAN["cic_ids2017"], MAN["cic_ids2017"], SCHEMA["cic_ids2017"], PREP)
v.insert(0, "dataset", "cic_ids2017"); v.to_csv(RES_DIR / "cic_ids2017_preprocessor_validation.csv", index=False)
display(v)
assert v.passed.all(), "preprocessor validation failed - see table"

,dataset,check,passed,detail
0,cic_ids2017,fitted on the training split only,True,"1,539,015 rows fitted, train has 1,539,015"
1,cic_ids2017,input columns match schema,True,
2,cic_ids2017,output width = feature names,True,69 features
3,cic_ids2017,no NaN/inf (train sample),True,
4,cic_ids2017,no NaN/inf (validation sample),True,
5,cic_ids2017,numeric outputs bounded (quantile-normal),True,max |z| 5.20
6,cic_ids2017,validation distribution matches train,True,"median |mean diff| 0.006, median |std diff| 0.005"
7,cic_ids2017,deterministic transform,True,
8,cic_ids2017,save/load round trip identical,True,


## 3.8 · CICIDS-2017 transformation

In [ ]:
NAMES["cic_ids2017"], LABELS["cic_ids2017"], OUT["cic_ids2017"] = transform_splits(PRE["cic_ids2017"], CLEAN["cic_ids2017"], MAN["cic_ids2017"], SCHEMA["cic_ids2017"], READY_DIR / "cic_ids2017")

  train  1,539,015 rows x 69 features -> train.parquet (168.6 MB)
  val      329,790 rows x 69 features -> val.parquet (38.5 MB)
  test     329,790 rows x 69 features -> test.parquet (38.5 MB)


## 3.9 · CICIDS-2017 final validation

In [ ]:
fv, OVERLAP["cic_ids2017"] = final_validation("cic_ids2017", MAN["cic_ids2017"], NAMES["cic_ids2017"], LABELS["cic_ids2017"], OUT["cic_ids2017"], SCHEMA["cic_ids2017"], SPLIT["cic_ids2017"])
FINAL.append(fv); fv.to_csv(RES_DIR / "cic_ids2017_final_validation.csv", index=False)
display(fv); print("train/test identical feature rows:", OVERLAP["cic_ids2017"])
assert fv.passed.all(), "final validation failed - see table"
del CLEAN["cic_ids2017"]; gc.collect()

,dataset,check,passed,detail
0,cic_ids2017,train: row count matches manifest,True,"1,539,015"
1,cic_ids2017,train: no NaN/inf,True,
2,cic_ids2017,train: features stored as float32,True,
3,cic_ids2017,val: row count matches manifest,True,"329,790"
4,cic_ids2017,val: no NaN/inf,True,
5,cic_ids2017,val: features stored as float32,True,
6,cic_ids2017,test: row count matches manifest,True,"329,790"
7,cic_ids2017,test: no NaN/inf,True,
8,cic_ids2017,test: features stored as float32,True,
9,cic_ids2017,same columns in every split,True,


train/test identical feature rows: {'train_test_identical_feature_rows': 44, 'pct_of_test': 0.01}


0

## 3.10 · UNSW-NB15 preprocessor fitting

In [ ]:
PRE["unsw_nb15"], PRE_INFO["unsw_nb15"] = fit_preprocessor(CLEAN["unsw_nb15"], MAN["unsw_nb15"], SCHEMA["unsw_nb15"], PREP)
joblib.dump(PRE["unsw_nb15"], PRE_DIR / "unsw_nb15_preprocessor.joblib")
display(pd.Series({k_: v for k_, v in PRE_INFO["unsw_nb15"].items() if k_ != "train_rows_sha256"}, dtype=object).to_frame("unsw_nb15"))

,unsw_nb15
fit_rows,84377
fit_seconds,2.3
n_input_features,42
n_numeric,39
n_categorical,3
n_output_features,81
dropped_constant_in_train,[]


## 3.11 · UNSW-NB15 preprocessor validation

In [ ]:
v = validate_preprocessor(PRE["unsw_nb15"], PRE_INFO["unsw_nb15"], CLEAN["unsw_nb15"], MAN["unsw_nb15"], SCHEMA["unsw_nb15"], PREP)
v.insert(0, "dataset", "unsw_nb15"); v.to_csv(RES_DIR / "unsw_nb15_preprocessor_validation.csv", index=False)
display(v)
assert v.passed.all(), "preprocessor validation failed - see table"

,dataset,check,passed,detail
0,unsw_nb15,fitted on the training split only,True,"84,377 rows fitted, train has 84,377"
1,unsw_nb15,input columns match schema,True,
2,unsw_nb15,output width = feature names,True,81 features
3,unsw_nb15,no NaN/inf (train sample),True,
4,unsw_nb15,no NaN/inf (validation sample),True,
5,unsw_nb15,numeric outputs bounded (quantile-normal),True,max |z| 5.20
6,unsw_nb15,validation distribution matches train,True,"median |mean diff| 0.020, median |std diff| 0.012"
7,unsw_nb15,unseen categories handled,True,
8,unsw_nb15,deterministic transform,True,
9,unsw_nb15,save/load round trip identical,True,


## 3.12 · UNSW-NB15 transformation

In [ ]:
NAMES["unsw_nb15"], LABELS["unsw_nb15"], OUT["unsw_nb15"] = transform_splits(PRE["unsw_nb15"], CLEAN["unsw_nb15"], MAN["unsw_nb15"], SCHEMA["unsw_nb15"], READY_DIR / "unsw_nb15")

  train     84,377 rows x 81 features -> train.parquet (6.7 MB)
  val       14,891 rows x 81 features -> val.parquet (1.2 MB)
  test      82,332 rows x 81 features -> test.parquet (4.9 MB)


## 3.13 · UNSW-NB15 final validation

In [ ]:
fv, OVERLAP["unsw_nb15"] = final_validation("unsw_nb15", MAN["unsw_nb15"], NAMES["unsw_nb15"], LABELS["unsw_nb15"], OUT["unsw_nb15"], SCHEMA["unsw_nb15"], SPLIT["unsw_nb15"])
FINAL.append(fv); fv.to_csv(RES_DIR / "unsw_nb15_final_validation.csv", index=False)
display(fv); print("train/test identical feature rows:", OVERLAP["unsw_nb15"])
assert fv.passed.all(), "final validation failed - see table"
del CLEAN["unsw_nb15"]; gc.collect()

,dataset,check,passed,detail
0,unsw_nb15,train: row count matches manifest,True,"84,377"
1,unsw_nb15,train: no NaN/inf,True,
2,unsw_nb15,train: features stored as float32,True,
3,unsw_nb15,val: row count matches manifest,True,"14,891"
4,unsw_nb15,val: no NaN/inf,True,
5,unsw_nb15,val: features stored as float32,True,
6,unsw_nb15,test: row count matches manifest,True,"82,332"
7,unsw_nb15,test: no NaN/inf,True,
8,unsw_nb15,test: features stored as float32,True,
9,unsw_nb15,same columns in every split,True,


train/test identical feature rows: {'train_test_identical_feature_rows': 5162, 'pct_of_test': 6.27}


0

### Module 3 manifest & summary

In [ ]:
manifest = {"created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "module": "3 - preprocessing & splits",
            "split": SPLIT, "preprocessing": PREP, "sklearn": sklearn.__version__, "datasets": {}}
lines = ["# Module 3 — Preprocessing & splits summary", "",
         f"Scaler: {PREP['numeric_scaler']}; one-hot cap {PREP['max_categories']}; seed {SEED}. All preprocessors fitted on the training split only.", "",
         "| Dataset | Train | Val | Test | Input features | Model features | Attack rate (train) | Train/test identical rows |",
         "|---|---|---|---|---|---|---|---|"]
for k in DATASETS:
    c = MAN[k].split.value_counts()
    manifest["datasets"][k] = {
        "split_manifest": str((SPLIT_DIR / f"{k}_split_manifest.parquet").relative_to(BASE)),
        "preprocessor": str((PRE_DIR / f"{k}_preprocessor.joblib").relative_to(BASE)),
        "preprocessor_sha256": sha256_file(PRE_DIR / f"{k}_preprocessor.joblib"),
        "files": {s: {"path": str(OUT[k][s]["path"].relative_to(BASE)), "rows": OUT[k][s]["rows"],
                      "sha256": sha256_file(OUT[k][s]["path"])} for s in ["train", "val", "test"]},
        "feature_names": NAMES[k], "label_columns": LABELS[k], "fit": PRE_INFO[k], "train_test_overlap": OVERLAP[k]}
    lines.append(f"| {k} | {c['train']:,} | {c['val']:,} | {c['test']:,} | {PRE_INFO[k]['n_input_features']} | {len(NAMES[k])} | "
                 f"{MAN[k].loc[MAN[k].split=='train','y_binary'].mean():.3f} | {OVERLAP[k]['train_test_identical_feature_rows']:,} ({OVERLAP[k]['pct_of_test']}%) |")
(RES_DIR / "module3_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
pd.concat(FINAL, ignore_index=True).to_csv(RES_DIR / "final_validation_all.csv", index=False)
(RES_DIR / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# Module 3 — Preprocessing & splits summary

Scaler: quantile; one-hot cap 20; seed 42. All preprocessors fitted on the training split only.

| Dataset | Train | Val | Test | Input features | Model features | Attack rate (train) | Train/test identical rows |
|---|---|---|---|---|---|---|---|
| bot_iot | 632,744 | 135,589 | 135,589 | 20 | 46 | 0.990 | 3 (0.0%) |
| cic_ids2017 | 1,539,015 | 329,790 | 329,790 | 69 | 69 | 0.153 | 44 (0.01%) |
| unsw_nb15 | 84,377 | 14,891 | 82,332 | 42 | 81 | 0.480 | 5,162 (6.27%) |
